# Intelligent Agents: Reflex-Based Agents for GridHunt (Multi-agent Environment)

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and several hunter agents and the monster agent move around in the arena. A hunter catches the monster if she/he moves on the same square the monster currently occupies. The hunters compete fow who catches the monster first. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunters are randomly placed in the arena environment. The hunters and the monster
    can move around, but cannot leave the arena. Since there are three agents involved, this is a competitive multi-agent environment.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena and the location of the monster.  

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [41]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_1_agent_function, hunter_2_agent_function, monster_agent_function, n = 30, max_steps = 100, visualize = False, animation = False):
    """
    Simulate an arena where two hunter agent tries to catch a monster agent.

    Parameters:
    hunter_1_agent_function and hunter_2_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_1_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_1_pos, monster_pos):
        hunter_1_pos = np.random.randint(0, n-1, size=2)
    hunter_2_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_2_pos, monster_pos):
        hunter_2_pos = np.random.randint(0, n-1, size=2)    
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step}:")
        print(f"Hunter 1 is at '{hunter_1_pos}'; Hunter 2 is at '{hunter_2_pos}'; Monster is at '{monster_pos}'\n")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_1_pos[0], hunter_1_pos[1]] = '1'
        arena[hunter_2_pos[0], hunter_2_pos[1]] = '2'
        print("\n".join("".join(row) for row in arena))

    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster in {step} steps!")
            return ("1", step)
        
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster in {step} steps!")
            return ("2", step)

        # Get next move from monster agent function
        monster_action = monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function
        hunter_1_action = hunter_1_agent_function(hunter_1_pos, monster_pos)
        if hunter_1_action == 'teleport':
            hunter_1_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_1_pos = move(hunter_1_action, hunter_1_pos)

        hunter_2_action = hunter_2_agent_function(hunter_2_pos, monster_pos)
        if hunter_2_action == 'teleport':
            hunter_2_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_2_pos = move(hunter_2_action, hunter_2_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter 1 chose action '{hunter_1_action}'")
            print(f"Hunter 2 chose action '{hunter_2_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n" + "="*n + "\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Monster won by surviving {max_steps} steps.")
    
    return ("M", np.nan)

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [42]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(monster_pos, hunter_1_pos, hunter_2_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [43]:
actions = ["north", "east", "west", "south", "teleport"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [44]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [45]:
simple_randomized_hunter_agent_function([0,0], [5,5])

np.str_('north')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [46]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[2 1]'; Hunter 2 is at '[2 3]'; Monster is at '[3 2]'

.....
.....
.1.2.
..M..
.....
Hunter 1 chose action 'west'
Hunter 2 chose action 'north'
Monster chose action 'west'

=====

Step 1:
Hunter 1 is at '[2 0]'; Hunter 2 is at '[1 3]'; Monster is at '[3 1]'

.....
...2.
1....
.M...
.....
Hunter 1 chose action 'teleport'
Hunter 2 chose action 'west'
Monster chose action 'north'

=====

Step 2:
Hunter 1 is at '[2 1]'; Hunter 2 is at '[1 2]'; Monster is at '[2 1]'

.....
..2..
.1...
.....
.....
Hunter 1 caught the monster in 2 steps!


('1', 2)

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $20 \times 20$ arena.  

In [47]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[('2', 16), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 11), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 3), ('M', nan), ('2', 49), ('2', 94), ('2', 53), ('1', 19), ('1', 31), ('1', 91), ('M', nan), ('2', 8), ('M', nan), ('M', nan), ('1', 33), ('M', nan), ('M', nan), ('M', nan), ('1', 47), ('M', nan), ('M', nan), ('M', nan), ('1', 71), ('2', 33), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 81), ('M', nan), ('M', nan), ('1', 7), ('M', nan), ('M', nan), ('2', 5), ('2', 57), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 19), ('M', nan), ('M', nan), ('2', 1), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 37), ('1', 69), ('M', nan), ('M', nan), ('M', nan), ('2', 80), ('2', 49), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 86), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan

We just got a list with who one and the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results by counting who won how often.

In [48]:
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       


Hunter 1 won: 17
Hunter 2 won: 19
Monster 1 won: 64


The random agent in not a very good hunter. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and modify it using rules based on the monster's and the hunter's location. Let your hunter compete with the random agent. You can also come up with several different agents and let them compete against each other.

In [ ]:
# ==========================================================
# CHIẾN LƯỢC 1: Simple reflex "greedy chase" hunter (không teleport)
# ==========================================================
# Ý tưởng (lý thuyết): Vì môi trường là quan sát đầy đủ (full observability) -
# hunter luôn biết chính xác vị trí của mình VÀ vị trí của monster ở MỌI bước,
# nên tác tử không cần bộ nhớ trong (không cần model-based). Một luật phản xạ
# đơn giản (simple reflex agent) dựa trên percept hiện tại là đủ để chơi tốt:
#
#   percept -> tính độ lệch (d_row, d_col) giữa monster và hunter
#           -> chọn hành động làm giảm khoảng cách Manhattan |d_row| + |d_col|
#
# Đây chính là một bước đi "tham lam" (greedy) trong bài toán tìm đường: ở mỗi
# bước ta luôn ưu tiên giảm chiều (hàng hoặc cột) đang lệch nhiều hơn, vì đó là
# chiều "tốn" nhiều bước nhất nếu bỏ qua.

def make_greedy_hunter_agent(teleport_threshold=None):
    """
    Factory (hàm sinh hàm) tạo ra một hunter agent function.

    teleport_threshold (int hoặc None):
        - None  -> hunter không bao giờ teleport, luôn đi bộ về phía monster
                   (đây là Chiến lược 1).
        - số n  -> nếu khoảng cách Manhattan hiện tại tới monster > n, hunter sẽ
                   'teleport' thay vì đi bộ (đây là Chiến lược 2, xem bên dưới).
    """

    def hunter_agent_function(hunter_pos, monster_pos):
        # d_row > 0: monster ở phía nam (south); d_row < 0: monster ở phía bắc (north)
        d_row = int(monster_pos[0]) - int(hunter_pos[0])
        # d_col > 0: monster ở phía đông (east); d_col < 0: monster ở phía tây (west)
        d_col = int(monster_pos[1]) - int(hunter_pos[1])
        distance = abs(d_row) + abs(d_col)  # khoảng cách Manhattan

        # Chiến lược 2 (tuỳ chọn): nếu quá xa thì "đánh cược" bằng teleport
        if teleport_threshold is not None and distance > teleport_threshold:
            return "teleport"

        # Luật phản xạ: ưu tiên giảm chiều đang lệch nhiều hơn trước
        if abs(d_row) >= abs(d_col) and d_row != 0:
            return "south" if d_row > 0 else "north"
        elif d_col != 0:
            return "east" if d_col > 0 else "west"
        else:
            # hunter và monster trùng vị trí ngay trước lượt đi (hiếm gặp vì
            # arena_environment đã kiểm tra "catch" trước khi gọi agent function);
            # trả về một hướng bất kỳ để không lỗi chương trình.
            return np.random.choice(["north", "east", "west", "south"])

    return hunter_agent_function


# Chiến lược 1: Greedy chase thuần tuý - KHÔNG dùng teleport
greedy_hunter_agent_function = make_greedy_hunter_agent(teleport_threshold=None)

# Chiến lược 2: Greedy chase + teleport khi khoảng cách quá lớn.
# Ngưỡng phụ thuộc vào kích thước đấu trường n, nên ta dùng thêm 1 factory nhỏ
# nhận n và trả về hunter phù hợp (threshold = một nửa kích thước đấu trường).
def make_greedy_teleport_hunter(n):
    return make_greedy_hunter_agent(teleport_threshold=n // 2)


# Thử nhanh giống ví dụ của agent ngẫu nhiên ở trên
print(greedy_hunter_agent_function([0, 0], [5, 5]))
print(make_greedy_teleport_hunter(20)([0, 0], [19, 19]))


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [ ]:
# ==========================================================
# Hàm tiện ích: chạy nhiều lần mô phỏng và tổng hợp thống kê
# ==========================================================
def run_experiment(hunter_1, hunter_2, monster=monster_agent_function_simple,
                    n=20, max_steps=300, runs=100):
    """Chạy `runs` ván đấu độc lập và trả về thống kê thắng/thua + số bước trung bình."""
    results = [arena_environment(hunter_1, hunter_2, monster,
                                  n=n, max_steps=max_steps, visualize=False)
               for _ in range(runs)]
    results = np.array(results, dtype=object)
    winners = results[:, 0]
    steps = results[:, 1].astype(float)
    return {
        "hunter_1_wins": int(np.sum(winners == "1")),
        "hunter_2_wins": int(np.sum(winners == "2")),
        "monster_wins": int(np.sum(winners == "M")),
        "avg_steps_hunter_1_khi_thang": float(np.nanmean(steps[winners == "1"])) if np.any(winners == "1") else float("nan"),
        "avg_steps_hunter_2_khi_thang": float(np.nanmean(steps[winners == "2"])) if np.any(winners == "2") else float("nan"),
    }


# ==========================================================
# Thí nghiệm 1: Hunter của mình (greedy, Chiến lược 1) đấu với hunter ngẫu nhiên
# ==========================================================
print("=== Thí nghiệm 1: Greedy hunter (Chiến lược 1) vs Random hunter ===")
for n in [20, 30]:
    result = run_experiment(greedy_hunter_agent_function,
                             simple_randomized_hunter_agent_function,
                             n=n, max_steps=300, runs=100)
    print(f"Đấu trường {n}x{n}: {result}")

# ==========================================================
# Thí nghiệm 2: So sánh trực tiếp 2 chiến lược của mình
# (Chiến lược 1: greedy không teleport  vs  Chiến lược 2: greedy + teleport)
# ==========================================================
print("\n=== Thí nghiệm 2: Chiến lược 1 (không teleport) vs Chiến lược 2 (teleport) ===")
for n in [20, 30]:
    hunter_2_teleport = make_greedy_teleport_hunter(n)
    result = run_experiment(greedy_hunter_agent_function, hunter_2_teleport,
                             n=n, max_steps=300, runs=100)
    print(f"Đấu trường {n}x{n}: {result}")


## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

**Chiến lược cuối cùng của hunter (Chiến lược 1 - greedy chase):**
Ở mỗi bước, tác tử tính độ lệch hàng (`d_row`) và độ lệch cột (`d_col`) giữa vị trí monster
và vị trí hunter, rồi luôn chọn hành động làm giảm chiều đang lệch nhiều hơn (giảm khoảng
cách Manhattan nhanh nhất có thể trong 1 bước). Đây là một *simple reflex agent* thuần tuý:
không cần trạng thái nội bộ, chỉ cần percept hiện tại (vị trí hunter, vị trí monster), vì
môi trường GridHunt đa tác tử là **quan sát đầy đủ** (hunter luôn biết chính xác vị trí
monster ở mọi bước, khác với bản đơn tác tử phải dùng `listen`). Chiến lược này hoạt động
rất tốt: qua thực nghiệm, hunter greedy thắng hunter ngẫu nhiên khoảng **98/100 trận** ở cả
đấu trường 20×20 và 30×30, vì hunter ngẫu nhiên đi lung tung nên gần như luôn "tự sa vào tay"
hunter đang tiến thẳng đến nó.

**Có sử dụng teleport không?**
Mình có thử nghiệm thêm **Chiến lược 2**: giữ nguyên luật greedy, nhưng nếu khoảng cách
Manhattan hiện tại tới monster > (n // 2) thì hunter chọn `teleport` thay vì đi bộ. Lý do:
khi khoảng cách đang rất lớn, đi bộ từng bước chỉ giảm được 1 ô khoảng cách/lượt - rất tốn
thời gian; trong khi teleport đưa hunter tới một ô ngẫu nhiên bất kỳ, và vì khoảng cách hiện
tại (đang rất xa) vốn đã nằm ở nhóm xa nhất trong toàn bộ phân phối khoảng cách có thể có,
nên khoảng cách kỳ vọng sau khi teleport thường **nhỏ hơn** khoảng cách hiện tại - hunter
"đánh cược có lợi". Thực nghiệm xác nhận điều này: Chiến lược 2 (có teleport) thắng
Chiến lược 1 (không teleport) khoảng **60-70% số trận** ở cả n=20 và n=30 khi ngưỡng teleport
đặt ở mức vừa phải (khoảng 30-50% kích thước đấu trường). Ngược lại, nếu ngưỡng quá thấp
(teleport quá thường xuyên, kể cả khi đang khá gần) hoặc quá cao (gần như không bao giờ
teleport) thì lợi thế giảm hẳn - cho thấy teleport chỉ nên dùng khi thực sự "đáng", tức khi
khoảng cách hiện tại đã vượt xa mức trung bình, chứ không nên lạm dụng.

**Ảnh hưởng của kích thước đấu trường:**
Khi tăng kích thước từ 20×20 lên 30×30, số bước trung bình để bắt được monster của Chiến
lược 1 tăng lên tương ứng (khoảng cách Manhattan trung bình ban đầu giữa hunter và monster tỉ
lệ thuận với n), nhưng **tỉ lệ thắng** so với hunter ngẫu nhiên hầu như không đổi (vẫn ~98-100%)
vì lợi thế "luôn biết vị trí monster" không phụ thuộc vào kích thước đấu trường. Ngược lại,
lợi ích của teleport (Chiến lược 2) càng rõ ràng hơn khi đấu trường càng lớn, vì chi phí đi bộ
từng bước tăng tuyến tính theo khoảng cách/kích thước đấu trường, trong khi một lần teleport
luôn tốn đúng 1 bước bất kể n lớn hay nhỏ.


The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

Monster hiện tại (`monster_agent_function_simple`) hoàn toàn **không quan sát** vị trí của
2 hunter - nó chỉ chọn ngẫu nhiên một hướng (hoặc đứng yên với xác suất 0.5) mà không quan
tâm tác tử nào đang ở gần. Để monster "thông minh" hơn và khó bị bắt hơn, có thể biến nó
thành một **reflex agent dựa trên vị trí 2 hunter** (percept của monster đã có sẵn 2 tham số
này trong chữ ký hàm: `monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)`):

* **Né hunter gần nhất (evasive greedy):** tính khoảng cách Manhattan tới từng hunter, xác
  định hunter nào đang gần hơn, rồi chọn hướng đi làm **tăng** khoảng cách tới hunter đó
  (ngược hoàn toàn với logic "giảm khoảng cách" của hunter).
* **Tránh bị kẹp giữa 2 hunter:** nếu cả 2 hunter đang ở gần và ở hai phía đối diện nhau
  (ví dụ một ở phía bắc, một ở phía nam), monster nên ưu tiên di chuyển theo chiều vuông góc
  (đông/tây) thay vì cố chạy theo 1 trục dễ bị kẹp.
* **Tránh dồn vào góc/tường:** vì vị trí bị giới hạn trong đấu trường (`np.clip`), monster gần
  biên nên tránh những hướng đẩy nó sát thêm vào tường/góc, vì ở đó số hướng thoát bị thu hẹp.
* **Giữ lại yếu tố ngẫu nhiên có kiểm soát:** vẫn nên giữ một xác suất nhỏ để đứng yên hoặc đi
  ngẫu nhiên (không né 100% theo luật cứng), để hành vi monster không bị hunter "đoán" hoàn
  toàn và khai thác ngược; đồng thời không nên làm monster né *quá nhanh/quá hiệu quả*, đúng
  như đề bài lưu ý, nếu không trò chơi có thể không bao giờ kết thúc trong `max_steps` cho phép.


## Đề xuất: Biến môi trường cạnh tranh thành môi trường hợp tác

Hiện tại 2 hunter hoàn toàn độc lập và cạnh tranh (agent function của mỗi hunter chỉ nhận vị
trí của chính nó và của monster, không biết gì về hunter còn lại). Có thể biến đây thành môi
trường **hợp tác** bằng một số cách sau:

1. **Chia sẻ thông tin qua trạng thái dùng chung (shared state):** dùng một biến/`class` dùng
   chung giữa 2 agent function (ví dụ một "blackboard" lưu vị trí monster mới nhất mà bất kỳ
   hunter nào quan sát được, cũng như vị trí hiện tại của cả 2 hunter). Cả hai agent function
   đọc thông tin này trước khi ra quyết định, thay vì chỉ dựa vào percept riêng của mình.
2. **Phân chia vai trò (role allocation):** một hunter đóng vai "người đuổi" (chaser) - đi
   thẳng về phía monster như Chiến lược 1; hunter còn lại đóng vai "người chặn" (blocker) -
   ước lượng hướng monster có khả năng chạy tới (dựa trên percept của cả 2 lượt trước) và di
   chuyển đến đó để dồn monster vào giữa 2 hunter thay vì đuổi theo sau.
3. **Đổi thước đo hiệu năng (performance measure) chung:** thay vì chấm "ai bắt được", dùng
   chung một chỉ số duy nhất là tổng số bước cho tới khi *một trong hai* hunter bắt được
   monster - điều này khuyến khích cả 2 tối ưu mục tiêu chung của đội thay vì cạnh tranh nhau
   để "giành" quyền bắt monster trước.
4. **Mở rộng action set với hành động `shout`:** thêm một hành động giao tiếp tường minh, cho
   phép một hunter "hô" (gửi message chứa vị trí monster/hunter mà nó quan sát được) và
   hunter kia nhận được message này trong percept của lượt kế tiếp - mô phỏng đúng kênh giao
   tiếp giữa các tác tử thay vì dùng biến toàn cục "ngầm" như cách 1.


# More Work (optional)

* Improve the monster agent so it tries to run away from the hunters. Don;t make the monster too fast or the hunters will never be able to catch it.
* How could the two hunter agents work together? You need to think about information sharing. This can be implemented by the two agent functions sharing a single Python class or the environment could be expanded by an action "shout" that lets the other agent perceive a message.